In [ ]:
import sys
sys.path.append('..')

import torch
import json
import time
import numpy as np
import collections
from sklearn.metrics import classification_report, confusion_matrix
from torchvision import datasets
from torch.utils.data import Subset, DataLoader, random_split
import seaborn as sns
import matplotlib.pyplot as plt

from src.utils import get_quantum_device, set_seed
from src.dataset import get_dataloaders, get_transforms, DATA_DIR
from src.models import HybridModel
from src.train import train

set_seed(42)
device = get_quantum_device()
print(f"Device: {device}")

In [ ]:
model = HybridModel()
print(model)

dummy = torch.randn(2, 3, 64, 64)
out = model(dummy)
print(f"Output shape: {out.shape}")

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(batch_size=32, augment=True)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
small_train = Subset(train_loader.dataset, range(64))
small_val   = Subset(val_loader.dataset, range(32))
small_train_loader = DataLoader(small_train, batch_size=8, shuffle=True)
small_val_loader   = DataLoader(small_val,   batch_size=8, shuffle=False)

start = time.time()
history = train(model, small_train_loader, small_val_loader,
                epochs=1, lr=0.0005, device=device, save_name='hybrid_vqc_smoketest')
print(f"\nTime for 1 epoch on 64 samples: {time.time() - start:.1f}s")

In [ ]:
model.eval()
with torch.no_grad():
    images, labels = next(iter(small_val_loader))
    x = model.features(images)
    x = x.view(x.size(0), -1)
    x = torch.relu(model.fc(x))

    pre_out = model.vqc.pre(x)
    print("Pre-quantum (input to circuit) — first 5 samples:")
    print(pre_out[:5])

    q_out = torch.stack([model.vqc.qlayer(pre_out[i]) for i in range(pre_out.shape[0])])
    print("\nQuantum circuit output — first 5 samples:")
    print(q_out[:5])

In [ ]:
model = HybridModel()

start = time.time()
history = train(model, train_loader, val_loader,
                epochs=15, lr=0.0005, device=device, save_name='hybrid_vqc_fulldataset')
print(f"\nTotal time: {(time.time()-start)/3600:.2f} hours")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'])
ax1.set_title('Training Loss — 4-qubit VQC')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(history['val_acc'])
ax2.set_title('Validation Accuracy — 4-qubit VQC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')

plt.tight_layout()
plt.savefig('../results/vqc_4q_training_curves.png', dpi=150)
plt.show()

In [ ]:
model = HybridModel()
model.load_state_dict(torch.load('../checkpoints/hybrid_vqc_fulldataset_best.pth',
                                  map_location=device))
model.to(device)
model.eval()

classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway',
           'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

with open('../results/vqc_fulldataset_results.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes,
            yticklabels=classes, cmap='Purples')
plt.title('HybridModel (4-qubit VQC) Confusion Matrix — Full Dataset')
plt.tight_layout()
plt.savefig('../results/vqc_fulldataset_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
classical_acc = 0.93
vqc_acc = report['accuracy']

print(f"StrongCNN (classical):     {classical_acc:.4f}")
print(f"HybridModel (4-qubit VQC): {vqc_acc:.4f}")
print(f"Difference:                {vqc_acc - classical_acc:+.4f}")